In [ ]:
%%capture
!pip install llama-index transformers accelerate bitsandbytes pypdf
!pip install llama-index-llms-huggingface
!pip install langchain

In [ ]:
%%capture
!pip install -U langchain-community

In [ ]:
%%capture
!pip install llama-index-embeddings-langchain

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/preprocessed_article_df.csv"
preprocessed_article_df = pd.read_csv(path)

In [ ]:
preprocessed_article_df.head()

,id,companyName,published_at,articleUrl,companyUrl,title,Topic,article,sentiment_score
0,6854946,Kellogg,2023-05-19 08:24:00,https://www.kellogg.northwestern.edu/news/blog...,https://hackernoon.com/company/kellogg,Kellogg and Northwestern mourn the passing of ...,Edtech Undergraduate Education Programs,kellogg mba gateway global community countless...,0.515983
1,1083452,Enployable,2023-02-01 16:21:00,https://www.manilastandard.net/news/314301405/...,https://hackernoon.com/company/enployable,PBBM gives DepEd a year to revise K to 12 curr...,Edtech Undergraduate Education Programs,president ferdinand marcos jr gave department ...,0.521707
2,929566,Deutsche Telekom,2023-06-20 13:03:00,https://menafn.com/1106465035/EU-Push-To-Rip-A...,https://hackernoon.com/company/deutsche-telekom,EU Push To Rip And Replace Huawei 5G Meets Res...,Digital Project Management Solutions,remote attack producer firm namely huawei poss...,0.302314
3,2362136,POEM Technology LLC,2023-08-19 19:47:00,https://www.benzinga.com/pressreleases/23/08/n...,https://hackernoon.com/company/poemtechnologyllc,Technology-enabled Mobile Car Wash Service Was...,Electric Vehicle Charging System,washos llc washos company pleased announce mer...,0.653753
4,8776606,Seaboard,2022-12-03 07:57:00,https://www.koco.com/article/oklahoma-stitt-ex...,https://hackernoon.com/company/seaboard,Stitt files executive order to move state’s te...,Digital Project Management Solutions,information fusion center wa created central h...,0.224570


In [ ]:
documents = preprocessed_article_df.article

## Sentiment Analysis

In [ ]:
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
# textblob_polarity = documents.apply(lambda x: TextBlob(x).sentiment[0])
# vader_polarity = documents.apply(lambda x: SentimentIntensityAnalyzer().polarity_scores(x)['compound'])

In [ ]:
averaged_sentiment_score = (textblob_polarity + vader_polarity) / 2

In [ ]:
preprocessed_article_df['sentiment_score'] = averaged_sentiment_score

In [ ]:
# preprocessed_article_df.to_csv(path, index=False)

## Retrieve and Re-rank

In [ ]:
from datetime import datetime
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
def RetriveReRank(query, top_k=100):

  bi_encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
  bi_encoder.max_seq_length = 512     #Truncate long passages to 512 tokens
  top_k = top_k                          #Number of passages we want to retrieve with the bi-encoder

  question_embedding = bi_encoder.encode(query, convert_to_tensor=True)

  # corpus_embeddings = bi_encoder.encode(documents, convert_to_tensor=True, show_progress_bar=True)

  #The bi-encoder will retrieve 100 documents. We use a cross-encoder, to re-rank the results list to improve the quality
  cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

  corpus_embeddings = np.load("/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/corpus_embeddings.npy")
  corpus_embeddings = torch.from_numpy(corpus_embeddings)

  query = "What is the initiatives of cloud accounting software platform?"
  question_embedding = bi_encoder.encode(query, convert_to_tensor=True)
  question_embedding = question_embedding.cuda()
  hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k)
  hits = hits[0]  # Get the hits for the first query

  ##### Re-Ranking #####
  # Now, score all retrieved passages with the cross_encoder
  cross_inp = [[query, documents[hit['corpus_id']]] for hit in hits]
  cross_scores = cross_encoder.predict(cross_inp)
  mean_socres = np.mean(cross_scores)

  # Sort results by the cross-encoder scores
  for idx in range(len(cross_scores)):
    hits[idx]['cross-score'] = cross_scores[idx]

  hits = sorted(hits, key=lambda x: x['score'], reverse=True)
  hits = sorted(hits, key=lambda x: x['cross-score'], reverse=True)

  candidates = pd.DataFrame(hits)

  sentiment_score = preprocessed_article_df.loc[candidates["corpus_id"]].sentiment_score.values
  published_at = pd.to_datetime(preprocessed_article_df.loc[candidates["corpus_id"]].published_at.values)

  candidates["sentiment_score"] = sentiment_score
  candidates["published_at"] = published_at

  candidates.sort_values(by=["published_at", "sentiment_score", "cross-score"],
                       ascending=[False, False, False])

  difference = (max(candidates["published_at"]) - min(candidates["published_at"])) / 2

  threshold = min(candidates["published_at"]) + difference

  latest_candidates = candidates[
      (candidates["published_at"] > threshold) &
       (candidates["sentiment_score"] > 0) &
        (candidates["cross-score"] > mean_socres)]

  selected_documents = documents[latest_candidates["corpus_id"].values]
  return selected_documents

In [ ]:
# bi_encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
# bi_encoder.max_seq_length = 512     #Truncate long passages to 512 tokens
# top_k = 100                          #Number of passages we want to retrieve with the bi-encoder

# # corpus_embeddings = bi_encoder.encode(documents, convert_to_tensor=True, show_progress_bar=True)

# #The bi-encoder will retrieve 100 documents. We use a cross-encoder, to re-rank the results list to improve the quality
# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

In [ ]:
# corpus_embeddings = np.load("/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/corpus_embeddings.npy")
# corpus_embeddings = torch.from_numpy(corpus_embeddings)

In [ ]:
# query = "What is the initiatives of cloud accounting software platform?"
# question_embedding = bi_encoder.encode(query, convert_to_tensor=True)
# question_embedding = question_embedding.cuda()
# hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k)
# hits = hits[0]  # Get the hits for the first query

In [ ]:
# ##### Re-Ranking #####
#   # Now, score all retrieved passages with the cross_encoder
# cross_inp = [[query, documents[hit['corpus_id']]] for hit in hits]
# cross_scores = cross_encoder.predict(cross_inp)
# mean_socres = np.mean(cross_scores)

# # Sort results by the cross-encoder scores
# for idx in range(len(cross_scores)):
#   hits[idx]['cross-score'] = cross_scores[idx]

In [ ]:
# hits = sorted(hits, key=lambda x: x['score'], reverse=True)
# hits = sorted(hits, key=lambda x: x['cross-score'], reverse=True)

In [ ]:
# candidates = pd.DataFrame(hits)

In [ ]:
# from datetime import datetime
# sentiment_score = preprocessed_article_df.loc[candidates["corpus_id"]].sentiment_score.values
# published_at = pd.to_datetime(preprocessed_article_df.loc[candidates["corpus_id"]].published_at.values)

In [ ]:
# candidates["sentiment_score"] = sentiment_score
# candidates["published_at"] = published_at

In [ ]:
# candidates.sort_values(by=["published_at", "sentiment_score", "cross-score"],
#                        ascending=[False, False, False])

# difference = (max(candidates["published_at"]) - min(candidates["published_at"])) / 2

In [ ]:
# threshold = min(candidates["published_at"]) + difference

In [ ]:
# latest_candidates = candidates[(candidates["published_at"] > threshold) &
#  (candidates["sentiment_score"] > 0) &
#   (candidates["cross-score"] > mean_socres)]

In [ ]:
# selected_documents = documents[latest_candidates["corpus_id"].values]

## RAGs

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
from llama_index.core import VectorStoreIndex,SimpleDirectoryReader, PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import BitsAndBytesConfig
from llama_index.core.response.notebook_utils import display_response
from llama_index.core import Settings
from langchain.embeddings import HuggingFaceEmbeddings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in DeployedModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPI has co

In [ ]:
# documents = [Document(text=doc) for doc in selected_documents]
# documents = [Document(text=doc) for doc in documents]

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
def messages_to_prompt(messages):
  prompt = ""
  for message in messages:
    if message.role == 'system':
      prompt += f"<|system|>\n{message.content}\n"
    elif message.role == 'user':
      prompt += f"<|user|>\n{message.content}\n"
    elif message.role == 'assistant':
      prompt += f"<|assistant|>\n{message.content}\n"
# ensure we start with a system prompt, insert blank if needed
  if not prompt.startswith("<|system|>\n"):
    prompt = "<|system|>\n\n" + prompt

  # add final assistant prompt
  prompt = prompt + "<|assistant|>\n"

  return prompt

In [ ]:
llm = HuggingFaceLLM(
    model_name="meta-llama/Llama-2-7b-chat-hf",
    tokenizer_name="meta-llama/Llama-2-7b-chat-hf",
    query_wrapper_prompt=PromptTemplate("<|system|>\n\n<|user|>\n{query_str}\n<|assistant|>\n"),
    context_window=3900,
    max_new_tokens=512,
    model_kwargs={"quantization_config": quantization_config},
    # tokenizer_kwargs={},
    generate_kwargs={"temperature": 0.3, "top_k": 50, "top_p": 0.95},
    messages_to_prompt=messages_to_prompt,
    device_map="auto",

)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
local_model_path = "BAAI/bge-small-en-v1.5"
embed_model = HuggingFaceEmbeddings(model_name=local_model_path)

<ipython-input-17-da6e3d59c7b3>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name=local_model_path)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 3900

In [ ]:
def RAG(query, top_k=100):
  selected_documents = RetriveReRank(query, top_k)
  cand_documents = [Document(text=doc) for doc in selected_documents]

  index = VectorStoreIndex.from_documents(
    cand_documents, embed_model=embed_model
  )
  query_engine = index.as_query_engine(llm=llm)

  query = "<" + query + ">"
  response = query_engine.query(query)
  display_response(response)

In [ ]:
# index = VectorStoreIndex.from_documents(
#     documents, embed_model=embed_model
# )


# query_engine = index.as_query_engine(llm=llm)

In [ ]:
RAG("What is the initiatives of cloud accounting software platform?")

**`Final Response:`** Cloud accounting software platforms are designed to provide a range of initiatives that help small businesses manage their finances more efficiently. Some of the key initiatives of cloud accounting software platforms include:

1. Automating payroll system: Cloud accounting software platforms automate the payroll system, eliminating the need for manual data entry and reducing the chances of errors.
2. Time-saving: Cloud accounting software platforms provide enormous time-saving benefits to the finance department, saving up to 10,000 hours annually.
3. Integration with other business systems: Cloud accounting software platforms integrate with other business systems such as CRM, HR, and inventory management systems, providing a seamless workflow.
4. Customizable dashboard: Cloud accounting software platforms provide a customizable dashboard that allows users to present important information at a glance, tailoring the system to their existing workflow.
5. Invoicing and project management: Cloud accounting software platforms provide invoicing and project management tools, allowing users to track projects and invoices, and send reminders to clients.
6. Tax management: Cloud accounting software platforms provide tax management tools that help users accurately estimate their tax liability and save money accordingly.
7. Subscription pricing: Cloud accounting software platforms offer subscription pricing plans that are flexible and scalable, allowing users to choose the plan that best suits their business needs.
8. Free trial: Cloud accounting software platforms offer a free trial, allowing users to test the solution before committing to a purchase.

Overall, cloud accounting software platforms are designed to streamline business operations, reduce manual data entry, and provide valuable insights to help small businesses make informed financial decisions.

In [ ]:
RAG("Who are the biggest providers of cloud accounting software platform and what type is being used?")

**`Final Response:`** Based on the context information provided, the biggest providers of cloud accounting software platforms are:

1. QuickBooks: QuickBooks is one of the most popular cloud accounting software platforms used by small businesses. It offers a variety of features such as invoicing, expense tracking, inventory management, and payroll processing.
2. Xero: Xero is another popular cloud accounting software platform that is widely used by small businesses. It offers features such as invoicing, expense tracking, and project management, as well as integration with other business tools such as payment gateways and banking apps.
3. FreshBooks: FreshBooks is a cloud accounting software platform that is specifically designed for freelancers and small businesses. It offers features such as invoicing, time tracking, and expense tracking, as well as integration with other business tools such as payment gateways and project management software.
4. Zoho Books: Zoho Books is a cloud accounting software platform that offers features such as invoicing, expense tracking, and project management. It also integrates with other Zoho business tools such as CRM and inventory management.
5. Wave: Wave is a cloud accounting software platform that is designed for small businesses and freelancers. It offers features such as invoicing, expense tracking, and payment processing, as well as integration with other business tools such as payment gateways and banking apps.

All of these platforms are cloud-based, which means they can be accessed from anywhere and are easy to use. They also offer various pricing plans, making them accessible to small businesses of all sizes.

In [ ]:
RAG("<What are the strength, weaknesses, opportunity, and threats for cloud accounting software platform and each company?")

**`Final Response:`** Cloud accounting software platforms have several strengths, weaknesses, opportunities, and threats. Here are some of the key points to consider for each:

Strengths:

* Ease of use: Cloud accounting software platforms are designed to be user-friendly and easy to use, even for those without extensive accounting knowledge.
* Real-time reporting: Cloud accounting software provides real-time reporting, allowing businesses to track their financial performance and make informed decisions.
* Scalability: Cloud accounting software platforms can easily scale to meet the needs of growing businesses, without the need for expensive hardware upgrades.
* Collaboration: Cloud accounting software platforms allow for easy collaboration between team members, contractors, and clients, making it easier to work together on financial tasks.
* Automatic backups: Cloud accounting software platforms automatically backup data, ensuring that financial information is always available and protected.

Weaknesses:

* Security concerns: Some businesses may be hesitant to store sensitive financial information in the cloud due to security concerns.
* Dependence on internet connectivity: Cloud accounting software platforms require a stable internet connection, which can be a problem for businesses in areas with poor connectivity.
* Limited customization: While cloud accounting software platforms offer a range of features, they may not be fully customizable to meet the specific needs of each business.
* Cost: While cloud accounting software platforms are generally more affordable than traditional accounting software, they may still be a significant expense for some businesses.

Opportunities:

* Growing demand: As more businesses move to the cloud, the demand for cloud accounting software is likely to grow.
* Integration with other tools: Cloud accounting software platforms can easily integrate with other business tools, such as project management software and CRM systems.
* Mobile access: Cloud accounting software platforms offer mobile access, allowing businesses to manage their finances on the go.
* Expansion into new markets: Cloud accounting software platforms can be easily adapted for use in new markets, such as the accounting needs of freelancers or small businesses.

Threats:

* Competition: The cloud accounting software market is highly competitive, with many providers offering similar features and pricing.
* Regulatory changes

In [ ]:
RAG("What are the primary challenges or risks faced by clients when adopting or using a cloud accounting software platform?")

**`Final Response:`** The primary challenges or risks faced by clients when adopting or using a cloud accounting software platform include:

1. Data security and privacy concerns: Clients may worry about the safety of their financial data in the cloud, especially if they are not familiar with cloud technology.
2. Lack of control: Clients may feel that they have less control over their financial data when using a cloud accounting software platform compared to traditional on-premise accounting software.
3. Dependence on internet connectivity: Clients may worry about the potential for internet connectivity issues, which could impact their ability to access their financial data.
4. Compliance and regulatory issues: Clients may be concerned about the potential for non-compliance with regulatory requirements when using a cloud accounting software platform.
5. Integration with other systems: Clients may face challenges integrating their cloud accounting software with other business systems, such as payroll or inventory management software.
6. Cost and value: Clients may question the value of the cloud accounting software platform and the cost-benefit analysis of adopting the platform.
7. Lack of customization: Clients may feel limited in their ability to customize the cloud accounting software platform to meet their specific business needs.
8. Support and training: Clients may require additional support and training to effectively use the cloud accounting software platform.

By understanding these challenges and risks, clients can make informed decisions about adopting or using a cloud accounting software platform, and take steps to mitigate them.

In [ ]:
RAG("How much money is being invested in a cloud accounting software platform? How long are they anticipated to take?")

**`Final Response:`** Based on the information provided, it is anticipated that the cloud accounting software platform will take around 12 months to develop, with an estimated investment of $500,000. However, it is important to note that these estimates are based on the information provided and may not reflect the actual time and investment required for the development of the platform.

In [ ]:
RAG("What is the impact on sustainability metrics in a cloud accounting software platform? How will this impact jobs?")

**`Final Response:`** The impact of cloud accounting software on sustainability metrics is a crucial consideration, as it can significantly affect the environment and contribute to a more sustainable future. Here are some ways in which cloud accounting software can impact sustainability metrics:

1. Reduced carbon footprint: Cloud accounting software eliminates the need for physical hardware, such as servers and data centers, which can significantly reduce carbon emissions. According to a study by the University of California, the use of cloud computing can reduce carbon emissions by up to 30%.
2. Energy efficiency: Cloud accounting software is designed to be energy-efficient, using advanced technologies such as virtualization and server clustering to optimize energy consumption. This can lead to a reduction in energy consumption and lower carbon emissions.
3. Sustainable data centers: Cloud accounting software providers are investing in sustainable data centers, which are designed to minimize their environmental impact. These data centers use renewable energy sources, such as wind and solar power, to reduce their carbon footprint.
4. Remote work: Cloud accounting software enables remote work, which can reduce the need for commuting and lower carbon emissions. According to a study by the University of Cambridge, remote work can reduce carbon emissions by up to 50%.
5. Job creation: Cloud accounting software can create new job opportunities in areas such as software development, data analysis, and customer support. This can lead to an increase in jobs and contribute to economic growth while also promoting sustainability.

In conclusion, cloud accounting software can have a positive impact on sustainability metrics by reducing carbon emissions, optimizing energy consumption, investing in sustainable data centers, enabling remote work, and creating new job opportunities. This can contribute to a more sustainable future and promote economic growth while also supporting environmental protection.